In [1]:
!pip install pandas openpyxl requests

In [2]:
import pandas as pd
data = {"prompt": ["解释RAG","什么是AI Agent"]}
df = pd.DataFrame(data)
df.to_excel("test_prompt.xlsx", index=False)

In [3]:
#pandas：处理表格数据
#openpyxl：让 pandas 支持 Excel 读写
#requests：调用大模型接口
# ! 是 jupyter 专属语法，代表调用系统命令；普通 py 文件直接去掉！，用pip install xxx

import pandas as pd
import requests
import time

#配置区
API_KEY = "your API"
url = "https://dashscope.aliyuncs.com/api/v1/services/aigc/text-generation/generation" #大模型服务地址，也就是我们要发送请求的目标接口
INPUT_FILE = "test_prompt.xlsx"    #输入文件：存放所有待测试 Prompt 的 Excel
OUTPUT_FILE = "model_result.xlsx"
WAIT_SECONDS = 1                   #每调用一次接口，等待 1 秒，避免请求频率过高被服务商限制


#定义函数call_llm作用：接收一条 prompt，发起接口调用，返回模型回答、token、调用是否成功。
#参数 prompt_text 就是我们发给大模型的提问。


def call_llm(prompt_text):
    headers = {
        "Authorization":f"Bearer {API_KEY}",  #headers请求头。Authorization：带上密钥，证明身份；Content-Type：告诉接口，我们发送的数据是 JSON 格式。
        "Content-Type":"application/json"
    }
    body = {
        "model":"qwen-turbo",
        "input":{"messages":[{"role":"user","content":prompt_text}]},#请求体，传给大模型的核心参数：model：指定调用哪个模型；messages：对话结构，role:user 用户消息，content 就是提问。
        "parameters":{"result_format":"text"}
    }
    resp = requests.post(url,headers=headers,json=body,timeout=30)  #发起 POST 请求：向接口地址发送 headers 和 body。timeout=30，超过 30 秒没响应判定超时。
    res_json = resp.json()                                          #把接口返回的数据，转换成字典（JSON），方便提取内容。

    if res_json.get("output"):                           #判断接口是否正常返回结果。.get() 写法更安全，如果不存在 output 不会直接报错
        answer = res_json["output"]["text"]              #从返回 json 中提取大模型输出文本
        input_tokens = res_json["usage"]["input_tokens"]
        output_tokens = res_json["usage"]["output_tokens"]
        success = True
    else:
        answer = f"调用失败：{res_json}"
        input_tokens = 0
        output_tokens = 0
        success = False                               #如果接口异常，填充失败信息，token 置 0，标记调用失败。
    return answer,input_tokens,output_tokens,success  #函数返回 4 个数据：回答、输入 token、输出 token、成功标识。

if __name__ == "__main__":         #Python 标准写法：只有直接运行这个文件时，下面代码才执行
    df = pd.read_excel(INPUT_FILE) #pandas 读取 Excel 文件，把表格加载成 DataFrame（可以理解成程序里的虚拟表格）
    result_list = []               #创建空列表，用来存放每一条 prompt 的评测结果，最后统一写入 Excel

    for index,row in df.iterrows(): #循环遍历表格每一行，index：当前是第几行；row：当前这一行所有单元格数据
        prompt = row["prompt"]      #读取当前行中，列名为prompt单元格的内容，也就是测试提问
        print(f"正在测试:{prompt}")
        llm_ans,in_tok,out_tok,ok = call_llm(prompt) #调用上面写好的函数，传入 prompt，接收返回的四项结果

        bad_case = False                      #默认标记：不是坏样本
        if len(str(llm_ans)) < 15 or not ok:  #自定义评测规则：回答文本长度小于 15 个字 或者 接口调用失败 → 判定为 Bad Case。后续可扩展规则
            bad_case = True

        row_data = {                #用字典整理当前这条样本的所有数据，准备存入列表
            "prompt":prompt,
            "model_answer":llm_ans,
            "input_token":in_tok,
            "output_token":out_tok,
            "is_bad_case":bad_case
        }
        result_list.append(row_data) #把本条结果加入结果列表
        time.sleep(WAIT_SECONDS)     #暂停 1 秒，控制调用频率

    result_df = pd.DataFrame(result_list)       #把存放所有结果的列表，转换成 DataFrame 虚拟表格
    result_df.to_excel(OUTPUT_FILE,index=False) #保存到 Excel。index=False：不要导出自带的数字行号
    print(f"评测完成！结果保存在{OUTPUT_FILE}")

    bad_count = result_df["is_bad_case"].sum()  #对 is_bad_case 列求和，True=1，False=0，自动统计坏样本总数
    total_count = len(result_df)               #获取一共测试了多少条 prompt
    print(f"总测试样本:{total_count},BadCase数量:{bad_count}") #控制台输出汇总统计数据

正在测试:解释RAG
正在测试:什么是AI Agent
评测完成！结果保存在model_result.xlsx
总测试样本:2,BadCase数量:0


In [4]:
result_df  

,prompt,model_answer,input_token,output_token,is_bad_case
0,解释RAG,RAG 是 **Retrieval-Augmented Generation** 的缩写，中...,15,1139,False
1,什么是AI Agent,**AI Agent（人工智能代理）** 是一种能够自主感知环境、进行决策并采取行动的智能实...,15,901,False


In [12]:
import pandas as pd
import requests
import time

# 配置区
INPUT_FILE = "test_prompt.xlsx"
OUTPUT_FILE = "model_result.xlsx"
API_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions"
HEADERS = {
    "Content-Type": "application/json",
    "Authorization": "Bearer your API"
}
MODEL_NAME = "qwen-turbo"


# 读取prompt表格
df = pd.read_excel(INPUT_FILE)

# 创建输出字段
df["模型返回内容"] = ""
df["输入token数"] = ""
df["输出token数"] = ""
df["调用状态"] = "待执行"
df["评测标签"] = ""    # 正常 / 幻觉 / 逻辑错误 / 回答不相关
df["评测备注"] = ""
df["是否BadCase"] = ""  # 是 / 否

# 逐行循环调用
for index, row in df.iterrows():
    prompt = row["prompt"]
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7
    }

    try:
        resp = requests.post(API_URL, headers=HEADERS, json=payload, timeout=20)
        resp.raise_for_status()
        res_json = resp.json()

        answer = res_json["choices"][0]["message"]["content"]
        input_tokens = res_json["usage"]["prompt_tokens"]
        output_tokens = res_json["usage"]["completion_tokens"]

        df.at[index, "模型返回内容"] = answer
        df.at[index, "输入token数"] = input_tokens
        df.at[index, "输出token数"] = output_tokens
        df.at[index, "调用状态"] = "成功"

    except requests.exceptions.Timeout:
        # 请求超时
        df.at[index, "调用状态"] = "失败：接口请求超时"
    except requests.exceptions.ConnectionError:
        # 断网、域名无法访问、网络异常 触发这里
        df.at[index, "调用状态"] = "失败：网络异常/无法连接接口"
    except requests.exceptions.HTTPError as err:
        # 400/401/403/404/500 接口返回错误码
        df.at[index, "调用状态"] = f"失败：HTTP错误 {str(err)}"
    except ValueError:
        # 返回内容不是合法JSON
        df.at[index, "调用状态"] = "失败：接口返回JSON解析异常"
    except Exception as err:
        # 所有其余未知异常兜底，防止程序直接中断
        df.at[index, "调用状态"] = f"失败：未知异常 {str(err)}"

    time.sleep(0.8)

# 保存结果
df.to_excel(OUTPUT_FILE, index=False)
print("全部任务执行完成！结果已保存 model_result.xlsx")
print("人工打开Excel填写评测标签：正常/幻觉/逻辑错误/回答不相关")

# Jupyter预览
df

全部任务执行完成！结果已保存 model_result.xlsx
人工打开Excel填写评测标签：正常/幻觉/逻辑错误/回答不相关


,prompt,模型返回内容,输入token数,输出token数,调用状态,评测标签,评测备注,是否BadCase
0,解释RAG,RAG 是 **Retrieval-Augmented Generation** 的缩写，中...,15,1122,成功,,,
1,什么是AI Agent,**AI Agent（人工智能代理）** 是一种能够自主感知环境、进行决策并执行任务的智能实...,15,918,成功,,,


In [10]:
import pandas as pd
import requests
import time

# 配置区
INPUT_FILE = "test_prompt.xlsx"    
OUTPUT_FILE = "model_result.xlsx"  
API_URL = "https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions"
HEADERS = {
    "Content-Type": "application/json",
    "Authorization": "Bearer your API"
}
MODEL_NAME = "qwen-turbo" # 模型名称


# 1.读取测试Prompt表格
df = pd.read_excel(INPUT_FILE)

# === 原有接口结果字段 ===
df["模型返回内容"] = ""
df["输入token数"] = ""
df["输出token数"] = ""
df["调用状态"] = "待执行"

# 新增：多维度人工评测标记字段
df["评测标签"] = ""    # 正常 / 幻觉 / 逻辑错误 / 回答不相关
df["评测备注"] = ""    # 记录错误细节
df["是否BadCase"] = "" # 填：是 / 否

# 2.循环遍历每一行调用LLM接口
for index, row in df.iterrows():
    prompt = row["prompt"]  
    
    payload = {
        "model": MODEL_NAME,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": 0.7
    }

    try:
        resp = requests.post(API_URL, headers=HEADERS, json=payload, timeout=20)
        resp.raise_for_status()
        res_json = resp.json()

        answer = res_json["choices"][0]["message"]["content"]
        input_tokens = res_json["usage"]["prompt_tokens"]
        output_tokens = res_json["usage"]["completion_tokens"]

        df.at[index, "模型返回内容"] = answer
        df.at[index, "输入token数"] = input_tokens
        df.at[index, "输出token数"] = output_tokens
        df.at[index, "调用状态"] = "成功"

    except requests.exceptions.Timeout:
        df.at[index, "调用状态"] = "失败：接口请求超时"
    except requests.exceptions.ConnectionError:
        df.at[index, "调用状态"] = "失败：网络断开/无法连接接口"
    except requests.exceptions.HTTPError as err:
        df.at[index, "调用状态"] = f"失败：服务错误{str(err)}"
    except ValueError:
        df.at[index, "调用状态"] = "失败：接口返回数据格式异常"
    except Exception as err:
        df.at[index, "调用状态"] = f"失败：未知异常 {str(err)}"

    time.sleep(0.8)  

# 3.保存结果Excel
df.to_excel(OUTPUT_FILE, index=False)
print("全部任务执行完成！结果已保存")
print("提示：打开model_result.xlsx，在【评测标签】列填写：正常/幻觉/逻辑错误/回答不相关")

# Jupyter预览
df

全部任务执行完成！结果已保存
提示：打开model_result.xlsx，在【评测标签】列填写：正常/幻觉/逻辑错误/回答不相关


,prompt,模型返回内容,输入token数,输出token数,调用状态,评测标签,评测备注,是否BadCase
0,解释RAG,RAG 是 **Retrieval-Augmented Generation** 的缩写，中...,15,966,成功,,,
1,什么是AI Agent,**AI Agent（人工智能代理）** 是一种能够感知环境、自主决策并执行任务的智能系统。...,15,806,成功,,,


In [9]:
import requests

url = "https://dashscope.aliyuncs.com/compatible-mode/v1/chat/completions"
headers = {
    "Content-Type": "application/json",
    "Authorization": "Bearer your API"
}
payload = {
    "model": "qwen-turbo",
    "messages": [{"role": "user", "content": "你好"}],
    "temperature": 0.7
}
resp = requests.post(url, headers=headers, json=payload)
print("状态码：", resp.status_code)
print("返回详情：", resp.text)

状态码： 200
返回详情： {"choices":[{"finish_reason":"stop","index":0,"message":{"content":"你好！很高兴见到你。有什么我可以帮你的吗？😊","role":"assistant"}}],"created":1785439192,"id":"chatcmpl-d84d10fa-7806-9c82-9156-3285c5f36c7e","model":"qwen-turbo","object":"chat.completion","usage":{"completion_tokens":13,"prompt_tokens":13,"prompt_tokens_details":{"cached_tokens":0},"total_tokens":26}}
